# Making GPT from scratch

# Data

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-06-24 21:15:39--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2026-06-24 21:15:39 (130 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [2]:
# read
with open('../NanoGPT/input.txt', 'r', encoding='utf-8') as f:
  txt = f.read()

In [3]:
print(f'{len(txt):,d}')

1,115,394


In [4]:
print(txt[:20])

First Citizen:
Befor


In [5]:
print(txt[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [6]:
chars = sorted(list(set(txt)))

In [7]:
voc_sz = len(chars)
voc_sz, ' '.join(chars)

(65,
 "\n   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z")

In [8]:
stoi = {v:k for k,v in enumerate(chars)}
itos =  {k:v for k,v in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s]
decode = lambda i : ''.join([itos[n] for n in i])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [9]:
# decoding the data
import torch
data = torch.tensor(encode(txt), dtype=torch.long)
print(data.shape)
data[:100]

torch.Size([1115394])


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

`torch.long` = `64-bit integer`

In [10]:
print(decode(data[:100].tolist()))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [11]:
n = int(0.9 * len(data))
trn_ds = data[:n]
tst_ds = data[n:]

In [12]:
blk_sz = 8
trn_ds[:blk_sz+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
trn_ds[:blk_sz]

tensor([18, 47, 56, 57, 58,  1, 15, 47])

In [14]:
# andreij
x = trn_ds[:blk_sz]
y = trn_ds[1:blk_sz+1]
for i in range(blk_sz):
  a,b = x[:i+1],y[i]
  print(f'{a}----> {b}')

tensor([18])----> 47
tensor([18, 47])----> 56
tensor([18, 47, 56])----> 57
tensor([18, 47, 56, 57])----> 58
tensor([18, 47, 56, 57, 58])----> 1
tensor([18, 47, 56, 57, 58,  1])----> 15
tensor([18, 47, 56, 57, 58,  1, 15])----> 47
tensor([18, 47, 56, 57, 58,  1, 15, 47])----> 58


In [15]:
# mine
x = trn_ds[:blk_sz]
for i in range(1,blk_sz):
  a,b = x[:i],x[i]
  print(f'{a}----> {b}')

tensor([18])----> 47
tensor([18, 47])----> 56
tensor([18, 47, 56])----> 57
tensor([18, 47, 56, 57])----> 58
tensor([18, 47, 56, 57, 58])----> 1
tensor([18, 47, 56, 57, 58,  1])----> 15
tensor([18, 47, 56, 57, 58,  1, 15])----> 47


In [16]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    data = trn_ds if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y


xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print('targets:')
print(yb.shape)

inputs:
torch.Size([4, 8])
targets:
torch.Size([4, 8])


In [17]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [18]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):

  def __init__(self,voc_sz):
    super().__init__()
    self.token_embedding_table = nn.Embedding(voc_sz, voc_sz)

  def forward(self, idx, targ=None):
    logits = self.token_embedding_table(idx)
    if targ is None: loss = None
    else:
      b,t,c = logits.shape
      logits = logits.view(b*t,-1) # -1 is c
      targ  = targ.view(b*t)
      loss = F.cross_entropy(logits, targ)
    return logits, loss
  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(idx)
      logits = logits[:, -1, :] # (batch, 65 logits)
      probs = F.softmax(logits, dim=-1)   # (batch, 65 logits)
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

m = BigramLanguageModel(voc_sz)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(5.0364, grad_fn=<NllLossBackward0>)

lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


    We feed in a whole sequence of 8 chars at once:
    input:  [c1, c2, c3, c4, c5, c6, c7, c8]
            ↓    ↓    ↓    ↓    ↓    ↓    ↓   ↓
    output: [65, 65, 65, 65, 65, 65, 65, 65]  logits

    every position predicts its OWN next character simultaneously

    Why? Training efficiency:
    position 0: c1 → predicts c2
    position 1: c2 → predicts c3
    position 2: c3 → predicts c4
    ...
    position 7: c8 → predicts c9

    from ONE sequence of 8 chars
    we get 8 training examples at once ✅
    instead of feeding one char at a time ❌

    But at generation time:
    pythonlogits[:, -1, :]  # take ONLY last position

    ---------------------------------

    during generation idx keeps growing and we get logits for ALL positions but throw
    away everything except the last:
    iter 500: idx shape = (1, 500)
              logits shape = (1, 500, 65)  ← 500 predictions computed
              logits[:, -1, :]             ← only use position 499
              499 predictions thrown away  ← wasted computation ❌

    ---------------------------------
    # because we only care about predicting
    # the NEXT new character
    # the other positions already have their answers
    # in the actual sequence
    So T exists purely for training efficiency — get 8 predictions from 1 forward pass instead of 1.

    and that is for now using the BigramLanguageModel

In [19]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [20]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.362441062927246


In [21]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


M:
IUSh t,
F th he d ke alved.
Thupld, cipbll t
I: ir w, l me sie hend lor ito'l an e

I:
Gochosen ea ar btamandd halind
Aust, plt t wadyotl
I bel qunganonoth he m he de avellis k'l, tond soran:

WI he toust are bot g e n t s d je hid t his IAces I my ig t
Ril'swoll e pupat inouleacends-athiqu heamer te
Wht s

MI wect!-lltherotheve t fe;
WAnd py;

PO t s ld tathat, ir V
IO thesecin teot tit ado ilorer.
Ply, d'stacoes, ld omat mealellly yererer EMEvesas ie IZEd pave mautoofareanerllleyomerer but?


# SelfAttention

## Previous Means

In [22]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [23]:
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C) # we can say that t is the tokens
x.shape

torch.Size([4, 8, 2])

In [24]:
# Version 1
xbow = torch.zeros(B,T,C)
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1]
    xbow[b,t] = torch.mean(xprev, 0)

so that is we are taking the for each token in each batch the mean of the previous including itself and then putting it in xbow in inefficient way

In [25]:
# Version 2
wei = torch.tril(torch.ones(T, T))
wei /= wei.sum(1, keepdim=True)
xbow2 = wei @ x
torch.allclose(xbow, xbow2)

True

In [26]:
wei, x

(tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
         [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
         [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
         [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]]),
 tensor([[[-0.0431, -1.6047],
          [ 1.7878, -0.4780],
          [-0.2429, -0.9342],
          [-0.2483, -1.2082],
          [-0.7688,  0.7624],
          [-1.5673, -0.2394],
          [ 2.3228, -0.9634],
          [ 2.0024,  0.4664]],
 
         [[ 0.8008,  1.6806],
          [ 0.3559, -0.6866],
          [-0.4934,  0.2415],
          [-1.1109,  0.0915],
          [-0.2516,  0.8599],
     

`(...,M,N) @ (...,N,P)→(...,M,P)` ->
The last two dimensions are the actual matrix dims. Inner must match exactly. No exceptions, no broadcasting on these.

In [27]:
tril = torch.tril(torch.full((T, T), 5))
tril

tensor([[5, 0, 0, 0, 0, 0, 0, 0],
        [5, 5, 0, 0, 0, 0, 0, 0],
        [5, 5, 5, 0, 0, 0, 0, 0],
        [5, 5, 5, 5, 0, 0, 0, 0],
        [5, 5, 5, 5, 5, 0, 0, 0],
        [5, 5, 5, 5, 5, 5, 0, 0],
        [5, 5, 5, 5, 5, 5, 5, 0],
        [5, 5, 5, 5, 5, 5, 5, 5]])

In [28]:
# Version 3
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T,T)
wei = wei.masked_fill(tril==0, float('-inf'))
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [29]:
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [30]:
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

## Attention

In [31]:
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

head_sz = 16
kx = nn.Linear(C, head_sz, bias=False)
qx = nn.Linear(C, head_sz, bias=False)
vx = nn.Linear(C, head_sz, bias=False)

k,q,v = kx(x), qx(x), vx(x)
wei = k @ q.transpose(-1,-2)

trl = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

out = wei @ v
out.shape

torch.Size([4, 8, 16])

 `heads_sz = 16`-> not make `linear` `C->C`, but give use ability to make the model `expand the features` or `compress`, etc..


    What Was Happening in the Versions
    Version 2 the tril division gives you this weight matrix for T=4:
    1     0     0     0
    0.5   0.5   0     0
    0.33  0.33  0.33  0
    0.25  0.25  0.25  0.25
    Equal weight to every past token. Hard-coded.
    
    Version 3 starts from zeros then masks upper triangle with -inf then softmax.
              Softmax of equal values (all zeros) gives equal probabilities.
              So the output is identical to version 2 — still uniform average.
    Version 4 replaces those starting zeros with q @ k.T — actual numbers that
              differ per token pair. Now softmax over different numbers gives
              unequal weights. Some tokens matter more than others.

    The concrete reason uniform average is limiting: say your tokens are
    "the cat sat". When processing "sat", uniform average weights "the" and "cat"
    equally. But "cat" is far more relevant. Version 4 lets the model learn to weight
     "cat" much higher. That's the entire difference.

`version 2` is the previous mean methods that were before the `softmax method`

In [32]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4440, 0.5560, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6077, 0.1436, 0.2486, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6923, 0.0735, 0.1939, 0.0402, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0145, 0.4412, 0.0338, 0.0810, 0.4295, 0.0000, 0.0000, 0.0000],
        [0.0067, 0.2234, 0.0057, 0.1571, 0.2429, 0.3642, 0.0000, 0.0000],
        [0.0345, 0.0742, 0.2101, 0.1547, 0.0904, 0.1394, 0.2967, 0.0000],
        [0.0292, 0.1079, 0.0058, 0.0370, 0.1783, 0.0222, 0.0166, 0.6029]],
       grad_fn=<SelectBackward0>)

In [33]:
wei[0].sum(1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)

Notes:
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)


# Normalization For Attention
- "Scaled" attention additional divides `wei` by `1/sqrt(head_size)`. This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [34]:
k = torch.randn(B,T,head_sz)
q = torch.randn(B,T,head_sz)
wei = q @ k.transpose(-2, -1)
norm_wei =  q @ k.transpose(-2, -1) * head_sz**-0.5

In [35]:
k.var(), q.var()

(tensor(1.0116), tensor(0.9358))

In [36]:
wei.var(), norm_wei.var()

(tensor(14.3098), tensor(0.8944))

In [37]:
print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1))
print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1)) # gets too peaky, converges to one-hot
print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*80, dim=-1))
print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*800, dim=-1)) # turns to one hot encoded

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])
tensor([1.2664e-14, 4.7809e-25, 1.1254e-07, 4.7809e-25, 1.0000e+00])
tensor([0., 0., 0., 0., 1.])


In [38]:
class LayerNorm1d: # Averages over Features (BatchNorm is over Batches)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    xmean = x.mean(1, keepdim=True)
    xvar = x.var(1, keepdim=True)
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

In [39]:
torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [40]:
x[:,0].shape

torch.Size([32])

In [41]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [42]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

    # French to English translation example:

    # <--------- ENCODE ------------------><--------------- DECODE ----------------->
    # les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



# Putting It All Together as `.py` file
**This is a bit buggy and confusing implementation, So better implementation is in the implementation of GPT-2,3 in other file**


In [45]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('../NanoGPT/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)


epochs = 5000
bs = 64
lr = 3e-4
voc_sz = vocab_size
blk_sz = 128
n_emb = 256
head_sz = 64  # total, before splitting and dividing by n_heads (sorry for confusing naming)
n_heads = 8
dropout = 0.2
depth = 6        # how many transformers
eval_interval = 100  # for loss printing
eval_iters = 200     # for loss printing
device = 'cuda' if torch.cuda.is_available() else 'cpu'



# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split, device=device):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - blk_sz, (bs,))
    x = torch.stack([data[i:i+blk_sz] for i in ix])
    y = torch.stack([data[i+1:i+blk_sz+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out



class SelfAttention(nn.Module):
  def __init__(self, n_inp, head_sz, dropout):
    super().__init__()
    self.head_sz = head_sz
    self.key = nn.Linear(n_inp, head_sz, bias=False)
    self.query = nn.Linear(n_inp, head_sz, bias=False)
    self.value = nn.Linear(n_inp, head_sz, bias=False)
    self.proj = nn.Linear(head_sz, n_inp, bias=False)
    self.drop = nn.Dropout(dropout)
  def forward(self, x):
    b,t,c = x.shape
    k,q,v = self.key(x), self.query(x), self.value(x)
    wei = k @ q.transpose(-1,-2) * self.head_sz**-0.5 # Scaling
    trl = torch.tril(torch.ones(t,t))
    wei = wei.masked_fill(trl==0, float('-inf'))
    wei = F.softmax(wei, -1)
    return  self.drop(self.proj(wei @ v))

def batch_head(x, n_heads):
  b,t,c = x.shape
  x = x.reshape(b, t, n_heads,-1)
  return x.transpose(1,2).reshape(b*n_heads, t, -1)

def head_batch(x, n_heads):
    bn, t, c = x.shape
    x = x.reshape(-1, n_heads, t, c)
    return x.transpose(1, 2).reshape(-1, t, n_heads * c)


# This is a Bit confusing attention implementation and it has some bugs, Correct implementation is in the next implementation of GPT-2,3
class MultiHeadAttention(nn.Module):
  def __init__(self, n_inp, head_sz, n_heads, blk_sz, dropout):
    super().__init__()
    self.n_heads = n_heads
    self.head_sz, n_inp = head_sz//n_heads, n_inp//n_heads
    self.key = nn.Linear(n_inp, self.head_sz, bias=False)
    self.query = nn.Linear(n_inp, self.head_sz, bias=False)
    self.value = nn.Linear(n_inp, self.head_sz, bias=False)
    self.proj = nn.Linear(self.head_sz, n_inp, bias=False)
    self.drop = nn.Dropout(dropout)
    self.register_buffer('trl', torch.tril(torch.ones(blk_sz, blk_sz)))
  def forward(self, x):
    b,t,c = x.shape
    inp = x
    x = batch_head(x, self.n_heads) # BUG: attentions should happen before we make the batch head
    k,q,v = self.key(x), self.query(x), self.value(x)
    wei = k @ q.transpose(-1,-2) * self.head_sz**-0.5 # Scaling
    wei = wei.masked_fill(self.trl[:t,:t]==0, float('-inf'))
    wei = F.softmax(wei, -1)
    out = head_batch(self.proj(wei @ v), self.n_heads)
    return  self.drop(out)

class MLP(nn.Module):
  def __init__(self, n_inp, dropout):
    super().__init__()
    self.layers = nn.Sequential(nn.Linear(n_inp, 4 * n_inp),
                                nn.GELU(),
                                nn.Linear(4 * n_inp, n_inp),
                                nn.Dropout(dropout))
  def forward(self, x):
    return self.layers(x)

class Transformer(nn.Module):
  def __init__(self, n_inp, head_sz, n_heads, blk_sz, dropout):
    super().__init__()
    self.attn = MultiHeadAttention(n_inp, head_sz, n_heads, blk_sz, dropout)
    self.mlp = MLP(n_inp, dropout)
    self.norm1 = nn.LayerNorm(n_inp)
    self.norm2 = nn.LayerNorm(n_inp)
  def forward(self, x):
    x = x + self.attn(self.norm1(x))
    return x + self.mlp(self.norm2(x))



class GPT(nn.Module):

  def __init__(self,voc_sz, n_emb, blk_sz, head_sz, n_heads, dropout, depth=5):
    super().__init__()
    self.blk_sz = blk_sz
    self.Embs = nn.Embedding(voc_sz, n_emb)
    self.PosEmb = nn.Embedding(blk_sz, n_emb) # blk_sz is how much tokens model sees each time
    self.trans_layers = nn.Sequential(*[Transformer(n_emb, head_sz, n_heads, blk_sz, dropout) for i in range(depth)])
    self.norm = nn.LayerNorm(n_emb)
    self.head = nn.Linear(n_emb, voc_sz)

  def forward(self, idx, targ=None):
    b,t = idx.shape
    emb = self.Embs(idx) + self.PosEmb(torch.arange(t, device=device))
    out = self.trans_layers(emb)
    logits = self.head(self.norm(out))

    if targ is None: loss = None
    else:
      b,t,c = logits.shape
      logits = logits.view(b*t,-1) # -1 is c(voc_sz) -> (32, 65)
      targ  = targ.view(b*t)  # -> (32,)
      loss = F.cross_entropy(logits, targ)
    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens (because we have position Emb)
      idx_cond = idx[:, -self.blk_sz:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :] # (batch, 65 logits)
      probs = F.softmax(logits, dim=-1)   # (batch, 65 logits)
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx



model = GPT(voc_sz, n_emb, blk_sz, head_sz, n_heads, dropout, depth=depth)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for steps in range(epochs):

      # every once in a while evaluate the loss on train and val sets
    if steps % eval_interval == 0 or iter == epochs - 1:
        losses = estimate_loss()
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(decode(model.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=2000)[0].tolist()))

step 0: train loss 4.2979, val loss 4.2965
step 100: train loss 2.5099, val loss 2.5177
step 200: train loss 2.4752, val loss 2.4915
step 300: train loss 2.4608, val loss 2.4879
step 400: train loss 2.4504, val loss 2.4703
step 500: train loss 2.4271, val loss 2.4537
step 600: train loss 2.3959, val loss 2.4223
step 700: train loss 2.3501, val loss 2.3709
step 800: train loss 2.2823, val loss 2.3161
step 900: train loss 2.2228, val loss 2.2520
step 1000: train loss 2.1855, val loss 2.2188
step 1100: train loss 2.1464, val loss 2.1904
step 1200: train loss 2.1003, val loss 2.1561
step 1300: train loss 2.0561, val loss 2.1169
step 1400: train loss 2.0022, val loss 2.0825
step 1500: train loss 1.9543, val loss 2.0532
step 1600: train loss 1.9124, val loss 2.0171
step 1700: train loss 1.8679, val loss 1.9862
step 1800: train loss 1.8272, val loss 1.9655
step 1900: train loss 1.7862, val loss 1.9307
step 2000: train loss 1.7535, val loss 1.9067
step 2100: train loss 1.7214, val loss 1.8846


    += modifies x in place which destroys the information PyTorch needs to compute gradients. Always use x = x + ... instead
     of x += ... inside neural network forward passes.

    CORRECT:

    def forward(self, x):
        x = x + self.attn(self.norm1(x))  # not +=
        return x + self.mlp(self.norm2(x))

    iteration 1:  t=1
      wei shape:      (1, 1, 1)
      self.trl:       (32, 32)  ← CRASH, shapes don't match ❌
      self.trl[:1,:1]: (1, 1)   ← matches ✅

    iteration 5:  t=5
      wei shape:      (1, 5, 5)
      self.trl:       (32, 32)  ← CRASH ❌
      self.trl[:5,:5]: (5, 5)   ← matches ✅